# Prompt Engineering with Local Open-Source LLMs
### Author: Dr. Aliasghar Khavasi (ChatGPT is used in preparation of the notebook)

This notebook covers a **hands-on tutorial** on Prompt Engineering with Local Open-Source LLMs.  
You will learn prompt engineering by actually running a **local open-source LLM**, changing prompts, changing system instructions, and comparing outputs.

### What this notebook emphasizes
- prompt engineering as a practical skill,
- local inference with open-source models,
- model selection for different tasks,
- prompt styles such as **zero-shot**, **few-shot**, and **chain-of-thought-style prompting**,
- iterative prompt refinement,
- output evaluation, hallucination awareness, and bias awareness.

### What makes this notebook interactive
You will be able to:
1. choose a model from a curated list,
2. set your own **system instruction**,
3. write your own **user prompt**,
4. change generation settings,
5. compare multiple prompt variants,
6. and inspect how small wording changes affect outputs.

## Learning objectives

By the end of this notebook, you should be able to:

1. explain why prompt engineering matters for LLM behavior,
2. describe the roles of **context**, **style**, and **role/persona** in prompts,
3. distinguish between **zero-shot**, **few-shot**, and **chain-of-thought-style** prompting,
4. load and run a local open-source model with the Hugging Face `transformers` library,
5. compare several open-source instruct models and choose one based on task needs,
6. evaluate outputs qualitatively for relevance, clarity, faithfulness, and bias,
7. iteratively refine prompts to improve results,
8. use system instructions to steer model behavior in a controlled way.

## How this tutorial connects to the previous session

In the previous generative AI session, you studied how models such as autoencoders, VAEs, GANs, and transformers work, as well as how to build a LLM from scratch.

This session moves from **how models are built** to **how we use existing models well**.

A useful way to think about the transition is:

- previous session: **how generative models learn**
- this session: **how to communicate with an already trained LLM**

## A practical mental model for prompt engineering

Prompt engineering is not magic wording.  
It is closer to **interface design for language models**.

A good prompt does at least one of these clearly:

- sets the **task**,
- sets the **context**,
- sets the **role/persona**,
- sets the **style or format**,
- sets the **constraints**,
- sets the **evaluation target**.

### Real-world analogy
Imagine asking someone for help.

A weak request is:

> "Write something about climate."

A stronger request is:

> "Suppose you are a science journalist. In 4 bullet points, summarize the main climate-policy changes in the EU for a general audience. Avoid jargon."

The second request gives the person:
- a role,
- a task,
- an audience,
- a format,
- and a constraint.

LLMs behave similarly.

## Before you start: environment notes

This notebook is designed for **local inference**.

It uses:
- `transformers`
- `torch`
- optionally `accelerate`
- optionally `bitsandbytes` for lower-memory loading on supported GPUs

Install what you need in your environment, for example:

```bash
pip install torch transformers accelerate sentencepiece
```

Optional for lower-memory GPU loading:
```bash
pip install bitsandbytes
```

### Important note about local models
Some models are small enough for CPU or modest GPUs.  
Others may require more RAM / VRAM.

If you get out-of-memory errors:
- choose a smaller model,
- lower `max_new_tokens`,
- or try quantization / CPU inference.

## Curated model list for this notebook

The notebook uses a curated set of **current local-friendly instruct models**.  
The list intentionally mixes:
- **general-purpose chat models**,
- **small lightweight models**,
- **code-oriented models**,
- and **models good for constrained local experiments**.

We will use the **same notebook interface** for all of them, so you can compare how model choice affects answers.

In [ ]:
# Core imports
import os
import textwrap
import warnings

import torch
import ipywidgets as widgets

from IPython.display import display, clear_output
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
    set_seed,
)

warnings.filterwarnings("ignore")
set_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

In [ ]:
# Model catalog
MODEL_CATALOG = {
    "Qwen2.5-3B-Instruct": {
        "hf_id": "Qwen/Qwen2.5-3B-Instruct",
        "task_fit": "Strong general-purpose instruct model; good default choice for chat, reasoning, structured answers, multilingual prompts.",
        "size_note": "About 3B parameters; relatively practical for local experimentation.",
        "requires_login": False,
        "access_note": "",
    },
    "Mistral-7B-Instruct-v0.3": {
        "hf_id": "mistralai/Mistral-7B-Instruct-v0.3",
        "task_fit": "Strong general-purpose instruct model; good for robust chat experiments and instruction following.",
        "size_note": "7B model; may need more RAM / VRAM.",
        "requires_login": False,
        "access_note": "",
    },
    "Phi-3.5-mini-instruct": {
        "hf_id": "microsoft/Phi-3.5-mini-instruct",
        "task_fit": "Compact and capable; useful for local experiments, concise instruction following, and smaller hardware setups.",
        "size_note": "Mini model family; often a good balance between size and capability.",
        "requires_login": False,
        "access_note": "",
    },
    "SmolLM2-1.7B-Instruct": {
        "hf_id": "HuggingFaceTB/SmolLM2-1.7B-Instruct",
        "task_fit": "Very compact option for lightweight local demos and classroom experiments.",
        "size_note": "1.7B model; among the easiest in this list to run locally.",
        "requires_login": False,
        "access_note": "",
    },
    "Qwen2.5-Coder-7B-Instruct": {
        "hf_id": "Qwen/Qwen2.5-Coder-7B-Instruct",
        "task_fit": "Code-focused model; useful for programming prompts, debugging, and technical explanations.",
        "size_note": "7B model; choose this when the main task is coding rather than general chat.",
        "requires_login": False,
        "access_note": "",
    },
    "TinyLlama-1.1B-Chat-v1.0": {
        "hf_id": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        "task_fit": "Ultra-light chat model; useful for quick experiments on very limited hardware.",
        "size_note": "1.1B model; one of the easiest options to run on CPU.",
        "requires_login": False,
        "access_note": "",
    },
#    "Llama-3.2-3B-Instruct": {
#       "hf_id": "meta-llama/Llama-3.2-3B-Instruct",
#        "task_fit": "Good assistant-style model for dialogue, rewriting, summarization, and general educational prompting.",
#        "size_note": "3B model; practical if your machine can handle gated access models.",
#        "requires_login": True,
#        "access_note": "This model requires Hugging Face login and your own access token. You may also need approved access to the model repository.",
#    },
#    "Gemma-2-2b-it": {
#        "hf_id": "google/gemma-2-2b-it",
#        "task_fit": "Lightweight instruction-tuned model; useful for experimentation on more limited hardware.",
#        "size_note": "2B model; access may require accepting a license.",
#        "requires_login": True,
#        "access_note": "This model may require Hugging Face login, your own access token, and license acceptance/access approval.",
#    },
#    "FLAN-T5-Base": {
#        "hf_id": "google/flan-t5-base",
#        "task_fit": "Instruction-tuned encoder-decoder model; useful for summarization, QA, and lightweight instruction tasks.",
#        "size_note": "Base-sized seq2seq model; relatively lightweight for experimentation.",
#    },
}

for name, info in MODEL_CATALOG.items():
    print(f"- {name}")
    print(f"  HF id: {info['hf_id']}")
    print(f"  Best for: {info['task_fit']}")
    print(f"  Notes: {info['size_note']}")
    if info["requires_login"]:
        print(f"  Access: {info['access_note']}")
    print()

## How to choose a model

A simple practical rule:

- choose **Qwen2.5-3B-Instruct** if you want a strong default for general prompting,
- choose **Mistral-7B-Instruct-v0.3** if you want a stronger general model and you have more memory,
- choose **Phi-3.5-mini-instruct** or **SmolLM2-1.7B-Instruct** if your machine is limited,
- choose **Qwen2.5-Coder-7B-Instruct** if your task is mostly coding,
- choose **TinyLlama-1.1B-Chat-v1.0** if you want a very small chat model for CPU experiments,
- uncomment/choose **Gemma-2-2b-it** if you want a smaller general instruction-tuned model,
- uncomment/choose **Llama-3.2-3B-Instruct** if you want a compact assistant-style model and already have access to it.
- uncomment/choose **FLAN-T5-Base** if you want a lightweight instruction-tuned model for summarization, QA, and other text-to-text tasks.

### Important concept
Model choice is part of prompt engineering in practice.

Why?

Because prompt quality depends not only on wording but also on:
- the model’s training,
- its instruction-tuning quality,
- its chat template,
- its preferred style,
- and its size/capability limits.

In [ ]:
# ==== GUI USER SETTINGS: model selection ====

model_selector = widgets.Dropdown(
    options=list(MODEL_CATALOG.keys()),
    value="SmolLM2-1.7B-Instruct",
    description="Model:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="650px"),
)

local_files_only_widget = widgets.Checkbox(
    value=False,
    description="Use local files only",
    indent=False,
)

use_4bit_widget = widgets.Checkbox(
    value=False,
    description="Try 4-bit loading if available",
    indent=False,
)

trust_remote_code_widget = widgets.Checkbox(
    value=True,
    description="Trust remote code if required",
    indent=False,
)

model_settings_note = widgets.HTML(
    value=(
        "<b>Note:</b> After changing the model selection, rerun the next two cells "
        "that configure loading and load the tokenizer/model."
    )
)

model_settings_html = widgets.HTML()

def _sync_model_settings(change=None):
    global SELECTED_MODEL_NAME, LOCAL_FILES_ONLY, USE_4BIT_IF_AVAILABLE, TRUST_REMOTE_CODE, MODEL_ID

    SELECTED_MODEL_NAME = model_selector.value
    LOCAL_FILES_ONLY = local_files_only_widget.value
    USE_4BIT_IF_AVAILABLE = use_4bit_widget.value
    TRUST_REMOTE_CODE = trust_remote_code_widget.value

    assert SELECTED_MODEL_NAME in MODEL_CATALOG, "Choose a model name from MODEL_CATALOG."
    MODEL_ID = MODEL_CATALOG[SELECTED_MODEL_NAME]["hf_id"]

    extra_notes = []

    if MODEL_CATALOG[SELECTED_MODEL_NAME].get("requires_login", False):
        extra_notes.append(MODEL_CATALOG[SELECTED_MODEL_NAME]["access_note"])

    if USE_4BIT_IF_AVAILABLE and DEVICE != "cuda":
        extra_notes.append("4-bit loading requires CUDA, so it will not be used on CPU.")

    notes_html = ""
    if extra_notes:
        notes_html = "<br><br><b>Notes:</b><ul>" + "".join(f"<li>{note}</li>" for note in extra_notes) + "</ul>"

    model_settings_html.value = f"""
    <div style="
        width:650px;
        padding:12px 14px;
        border:1px solid #d9d9d9;
        border-radius:10px;
        background:#fafafa;
        line-height:1.6;
        box-sizing:border-box;
    ">
        <b>Selected model:</b> {SELECTED_MODEL_NAME}<br>
        <b>Model id:</b> {MODEL_ID}<br>
        <b>Task fit:</b> {MODEL_CATALOG[SELECTED_MODEL_NAME]['task_fit']}<br>
        <b>Size note:</b> {MODEL_CATALOG[SELECTED_MODEL_NAME]['size_note']}<br>
        <b>Use local files only:</b> {LOCAL_FILES_ONLY}<br>
        <b>Try 4-bit loading:</b> {USE_4BIT_IF_AVAILABLE}<br>
        <b>Trust remote code:</b> {TRUST_REMOTE_CODE}
        {notes_html}
    </div>
    """

for _w in [model_selector, local_files_only_widget, use_4bit_widget, trust_remote_code_widget]:
    _w.observe(_sync_model_settings, names="value")

_sync_model_settings()

display(
    widgets.VBox([
        model_selector,
        widgets.HBox([local_files_only_widget, use_4bit_widget, trust_remote_code_widget]),
        model_settings_note,
        model_settings_html,
    ])
)

In [ ]:
# Optional 4-bit config
bnb_config = None

if USE_4BIT_IF_AVAILABLE:
    if DEVICE == "cuda":
        try:
            from transformers import BitsAndBytesConfig
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.float16,
            )
            print("4-bit loading is enabled.")
        except Exception as e:
            print("Could not enable 4-bit loading:", e)
            bnb_config = None
    else:
        print("4-bit loading requested, but CUDA is not available. Using standard model loading.")
else:
    print("Using standard model loading.")

## Optional: Hugging Face Login (Recommended but not required)

In most cases, you can run this notebook without logging in to Hugging Face.  
However, logging in can improve your experience and is sometimes necessary.

### When is login NOT required?
- When using **public, non-gated models**
- When doing **small-scale experiments**
- When you are okay with slower downloads or occasional rate limits

### When SHOULD you log in?
- When using **restricted (gated) models** such as Llama or Gemma
- When you want **faster downloads and higher rate limits**
- When working with **larger models or repeated downloads**

### What is used for login?
Hugging Face uses an **access token** (not your password).  
This token securely identifies you and allows access to models you are অনুমitted to use.

### How to get your token
1. Go to: https://huggingface.co/settings/tokens  
2. Click **"New token"**  
3. Select role: **Read**  
4. Copy the generated token  

### Important notes
- Keep your token **private** (do not share it publicly)
- You only need to log in **once per environment/session**
- Login does **not automatically grant access** to gated models — you must request access separately on the model page

---

If you want, you can log in below using your token. Otherwise, you can skip this step and continue.

In [ ]:
# ==== Hugging Face login helper (GUI inside notebook) ====

from huggingface_hub import login
import ipywidgets as widgets
from IPython.display import display, clear_output

hf_token_input = widgets.Password(
    description="HF Token:",
    placeholder="Paste your Hugging Face token",
    layout=widgets.Layout(width="600px"),
    style={"description_width": "initial"},
)

hf_login_button = widgets.Button(
    description="Login to Hugging Face",
    button_style="primary",
)

hf_logout_button = widgets.Button(
    description="Clear token field",
    button_style="warning",
)

hf_login_output = widgets.Output()

def handle_hf_login(b):
    with hf_login_output:
        clear_output()
        token = hf_token_input.value.strip()

        if not token:
            print("No token entered.")
            return

        try:
            login(token=token)
            print("Successfully logged in to Hugging Face.")
        except Exception as e:
            print("Login failed:", e)

def handle_clear_token(b):
    hf_token_input.value = ""
    with hf_login_output:
        clear_output()
        print("Token field cleared.")

hf_login_button.on_click(handle_hf_login)
hf_logout_button.on_click(handle_clear_token)

display(
    widgets.VBox([
        widgets.HTML("<b>Hugging Face Login</b>"),
        hf_token_input,
        widgets.HBox([hf_login_button, hf_logout_button]),
        hf_login_output,
    ])
)

In [ ]:
# Load tokenizer and model

model = None
tokenizer = None

HF_TOKEN = os.getenv("HF_TOKEN", None)

if HF_TOKEN:
    print("Using HF_TOKEN from environment.")
else:
    print("No HF_TOKEN environment variable found.")
    print("If you already logged in through the notebook login helper, that is okay.")

try:
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        local_files_only=LOCAL_FILES_ONLY,
        trust_remote_code=TRUST_REMOTE_CODE,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs = {
        "local_files_only": LOCAL_FILES_ONLY,
        "trust_remote_code": TRUST_REMOTE_CODE,
    }

    if bnb_config is not None:
        model_kwargs["quantization_config"] = bnb_config
        model_kwargs["device_map"] = "auto"
    else:
        if DEVICE == "cuda":
            model_kwargs["dtype"] = torch.float16
            model_kwargs["device_map"] = "auto"
        else:
            model_kwargs["dtype"] = torch.float32

    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)
    model.eval()

    print("Model loaded successfully.")

except Exception as e:
    error_text = str(e)

    print(f"Failed to load model: {MODEL_ID}")
    print(f"Error type: {type(e).__name__}")
    print(error_text)

    error_lower = error_text.lower()

    if "403" in error_lower or "not in the authorized list" in error_lower:
        print("\nYou appear to be logged in, but you do not have approved access to this gated model yet.")
        print("Visit the model page, request access, wait for approval if required, then try again.")

    elif "gated repo" in error_lower or "401" in error_lower or "please log in" in error_lower:
        print("\nThis model requires Hugging Face login and may also require approved access.")
        print("Log in with your Hugging Face token, then try again.")

    model = None
    tokenizer = None

## Why we use chat templates

Most modern instruct models are still **causal language models** underneath.  
They continue a token sequence.

The difference is that chat models expect input in a specific **message format**.

That is why we use:

```python
tokenizer.apply_chat_template(...)
```

### Why this matters
Different instruct models expect different control tokens and formatting.

So instead of manually inventing the prompt wrapper, we let the tokenizer format the conversation the way the model expects.

In [ ]:
def build_messages(system_instruction, user_prompt):
    messages = []
    if system_instruction and system_instruction.strip():
        messages.append({"role": "system", "content": system_instruction.strip()})
    messages.append({"role": "user", "content": user_prompt.strip()})
    return messages


def generate_from_chat(
    model,
    tokenizer,
    system_instruction,
    user_prompt,
    max_new_tokens=220,
    temperature=0.7,
    top_p=0.95,
    top_k=50,
    do_sample=True,
    repetition_penalty=1.05,
):
    messages = build_messages(system_instruction, user_prompt)

    # Safe default generation settings used internally
    max_new_tokens = 220
    temperature = 0.7
    top_p = 0.95
    top_k = 50
    do_sample = True
    repetition_penalty = 1.05

    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template is not None:
        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        )
    else:
        prompt_parts = []
        if system_instruction and system_instruction.strip():
            prompt_parts.append(f"System: {system_instruction.strip()}")
        prompt_parts.append(f"User: {user_prompt.strip()}")
        prompt_parts.append("Assistant:")
        prompt_text = "\n\n".join(prompt_parts)

        inputs = tokenizer(
            prompt_text,
            return_tensors="pt",
            padding=True,
            truncation=True,
        )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    input_length = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
            top_p=top_p if do_sample else None,
            top_k=top_k if do_sample else None,
            repetition_penalty=repetition_penalty,
            pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][input_length:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response.strip()

# 1. Prompt Engineering Basics

We begin with the three ideas that influence model behavior the most:

1. **Context**  
2. **Style**  
3. **Role / Persona**

## 1.1 Context

**Context** gives the model the background it needs to interpret the request.

### Weak prompt
> "Summarize this."

### Better prompt
> "Summarize the following article about renewable energy in 3 bullet points for first-year engineering students."

The better prompt tells the model:
- what task to perform,
- what the topic is,
- how to format the answer,
- and who the answer is for.

### Analogy
Context is like handing someone the **question sheet plus the course name** instead of only asking a random question.

## 1.2 Style

**Style** controls tone, formality, audience level, and presentation.

You can ask for:
- formal,
- conversational,
- technical,
- beginner-friendly,
- concise,
- persuasive,
- creative,
- bullet-point format,
- table format,
- etc.

### Analogy
Style is like telling a speaker whether to talk:
- like a professor,
- like a journalist,
- like a tutor,
- or like a friend.

## 1.3 Role / Persona

A **role** tells the model what kind of voice or perspective to adopt.

Examples:
- "You are a patient tutor."
- "You are a cybersecurity analyst."
- "You are a career coach."
- "You are a strict reviewer."

### Why role helps
It changes:
- vocabulary,
- depth,
- tone,
- assumptions,
- and what kind of answer structure the model prefers.

### Analogy
Role is like assigning a costume and job to an actor before a scene begins.

In [ ]:
# ==== GUI USER SETTINGS: your main system instruction and prompt ====

import html
import time

Prompt_A_widget = widgets.Textarea(
    value=(
        "You are a university teaching assistant. "
        "Give accurate, structured, beginner-friendly answers to the following question: "
        "What is \"prompt engineering\"?"
    ),
    description="Prompt A:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="900px", height="110px"),
)

Prompt_B_widget = widgets.Textarea(
    value="Prompt eng.?",
    description="Prompt B:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="900px", height="120px"),
)

# Hidden advanced generation settings: using safe defaults in code instead
# max_new_tokens_widget = widgets.IntSlider(
#     value=220, min=32, max=1024, step=8, description="Max new tokens:",
#     style={"description_width": "initial"}, layout=widgets.Layout(width="500px")
# )
# temperature_widget = widgets.FloatSlider(
#     value=0.7, min=0.0, max=1.5, step=0.05, description="Temperature:",
#     style={"description_width": "initial"}, layout=widgets.Layout(width="500px")
# )
# top_p_widget = widgets.FloatSlider(
#     value=0.95, min=0.1, max=1.0, step=0.01, description="Top-p:",
#     style={"description_width": "initial"}, layout=widgets.Layout(width="500px")
# )
# top_k_widget = widgets.IntSlider(
#     value=50, min=1, max=200, step=1, description="Top-k:",
#     style={"description_width": "initial"}, layout=widgets.Layout(width="500px")
# )
# do_sample_widget = widgets.Checkbox(value=True, description="Use sampling", indent=False)

run_main_prompt_button = widgets.Button(
    description="Run prompt",
    button_style="primary",
    layout=widgets.Layout(width="160px", height="40px"),
)

run_status_html = widgets.HTML(
    value="<span style='color:#666;'>Ready.</span>"
)

run_progress = widgets.IntProgress(
    value=0,
    min=0,
    max=100,
    description="Processing:",
    bar_style="",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px", visibility="hidden"),
)

run_elapsed_html = widgets.HTML(
    value="<span style='color:#666;'>Elapsed time: 0.0s</span>"
)

run_spinner_html = widgets.HTML(
    value="",
    layout=widgets.Layout(width="900px")
)

response_html = widgets.HTML(
    value="""
    <div style="
        width:900px;
        min-height:120px;
        padding:14px 16px;
        border:1px solid #d9d9d9;
        border-radius:10px;
        background:#fafafa;
        color:#444;
        line-height:1.5;
        box-sizing:border-box;
    ">
        Responses will appear here.
    </div>
    """
)

def _format_response_box(title, text, border_color="#d9d9d9", background="#fafafa"):
    safe_title = html.escape(title)
    safe_text = html.escape(text).replace("\n", "<br>")
    return f"""
    <div style="
        width:100%;
        padding:16px 18px;
        border:1px solid {border_color};
        border-radius:10px;
        background:{background};
        line-height:1.6;
        white-space:normal;
        overflow-wrap:anywhere;
        box-sizing:border-box;
    ">
        <div style="font-weight:700; margin-bottom:10px;">{safe_title}</div>
        <div style="white-space:pre-wrap;">{safe_text}</div>
    </div>
    """

def _format_two_response_layout(response_a, response_b):
    box_a = _format_response_box("Response to Prompt A", response_a, "#cfd8ff", "#f7f9ff")
    box_b = _format_response_box("Response to Prompt B", response_b, "#cfead6", "#f7fcf8")
    return f"""
    <div style="
        width:900px;
        display:grid;
        grid-template-columns:1fr;
        gap:14px;
        box-sizing:border-box;
    ">
        {box_a}
        {box_b}
    </div>
    """

def _running_spinner_html():
    return """
    <div style="display:flex; align-items:center; gap:10px; color:#1f77b4; margin:6px 0 4px 0;">
        <div style="
            width:18px;
            height:18px;
            border:3px solid #d0d7ff;
            border-top:3px solid #1f77b4;
            border-radius:50%;
            animation: spin 1s linear infinite;
        "></div>
        <div><b>Generating responses...</b></div>
    </div>

    <style>
    @keyframes spin {
        0% { transform: rotate(0deg); }
        100% { transform: rotate(360deg); }
    }
    </style>
    """

def _run_main_prompt(_=None):
    global Prompt_A, Prompt_B, MAX_NEW_TOKENS, TEMPERATURE, TOP_P, TOP_K, DO_SAMPLE

    Prompt_A = Prompt_A_widget.value
    Prompt_B = Prompt_B_widget.value

    # Safe internal defaults
    MAX_NEW_TOKENS = 220
    TEMPERATURE = 0.7
    TOP_P = 0.95
    TOP_K = 50
    DO_SAMPLE = True

    start_time = time.perf_counter()

    run_main_prompt_button.disabled = True
    run_main_prompt_button.description = "Running..."
    run_status_html.value = (
        "<span style='color:#1f77b4;'><b>Running...</b> "
        "Please wait while the model generates the responses.</span>"
    )
    run_progress.layout.visibility = "visible"
    run_progress.bar_style = "info"
    run_progress.value = 20
    run_elapsed_html.value = "<span style='color:#1f77b4;'>Elapsed time: running...</span>"
    run_spinner_html.value = _running_spinner_html()
    response_html.value = _format_response_box("Responses", "Generating responses...")

    if model is None or tokenizer is None:
        elapsed = time.perf_counter() - start_time
        run_status_html.value = (
            "<span style='color:#b22222;'><b>No model is currently loaded.</b> "
            "Please load a model first.</span>"
        )
        run_progress.bar_style = "danger"
        run_progress.value = 100
        run_elapsed_html.value = f"<span style='color:#b22222;'>Elapsed time: {elapsed:.1f}s</span>"
        run_spinner_html.value = ""
        response_html.value = _format_response_box(
            "Responses",
            "No model is currently loaded. Please load a model first.",
            border_color="#f0b4b4",
            background="#fff5f5",
        )
        run_main_prompt_button.disabled = False
        run_main_prompt_button.description = "Run prompt"
        return

    try:
        run_progress.value = 40

        response_a = generate_from_chat(
            model=model,
            tokenizer=tokenizer,
            system_instruction="You are a patient university teaching assistant.",
            user_prompt=Prompt_A,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            do_sample=DO_SAMPLE,
        )

        run_progress.value = 75

        response_b = generate_from_chat(
            model=model,
            tokenizer=tokenizer,
            system_instruction="You are a patient university teaching assistant.",
            user_prompt=Prompt_B,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            do_sample=DO_SAMPLE,
        )

        elapsed = time.perf_counter() - start_time

        run_progress.value = 100
        run_progress.bar_style = "success"
        run_status_html.value = (
            "<span style='color:green;'><b>Done.</b> Responses generated successfully.</span>"
        )
        run_elapsed_html.value = f"<span style='color:green;'>Elapsed time: {elapsed:.1f}s</span>"
        run_spinner_html.value = ""
        response_html.value = _format_two_response_layout(response_a, response_b)

    except Exception as e:
        elapsed = time.perf_counter() - start_time
        run_progress.value = 100
        run_progress.bar_style = "danger"
        run_status_html.value = "<span style='color:#b22222;'><b>Prompt run failed.</b></span>"
        run_elapsed_html.value = f"<span style='color:#b22222;'>Elapsed time: {elapsed:.1f}s</span>"
        run_spinner_html.value = ""
        response_html.value = _format_response_box(
            "Error",
            f"Prompt run failed.\n\nError type: {type(e).__name__}\n{str(e)}",
            border_color="#f0b4b4",
            background="#fff5f5",
        )

    finally:
        run_main_prompt_button.disabled = False
        run_main_prompt_button.description = "Run prompt"

run_main_prompt_button.on_click(_run_main_prompt)

display(
    widgets.VBox([
        Prompt_A_widget,
        Prompt_B_widget,
        widgets.HBox([run_main_prompt_button]),
        run_status_html,
        run_progress,
        run_elapsed_html,
        run_spinner_html,
        response_html,
    ])
)

# _run_main_prompt()

## 1.4 What changed because of the system instruction?

When you use a system instruction, you are setting the **initial operating rules** for the model.

Examples of what system instructions can influence:
- whether the answer is short or detailed,
- whether the tone is formal or conversational,
- whether it explains step by step,
- whether it should avoid jargon,
- whether it should admit uncertainty.

This is one of the most important parts of practical prompt engineering.

In [ ]:
# ==== Compare the same user prompt with two different Prompt A / Prompt B instructions ====

import html
import time

Comparison_User_Prompt_widget = widgets.Textarea(
    value="Explain transformers in a way a first-year student can understand.",
    description="User prompt:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="900px", height="110px"),
)

Comparison_Prompt_A_widget = widgets.Textarea(
    value="You are a concise technical tutor. Use short paragraphs and one analogy.",
    description="Prompt A:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="900px", height="95px"),
)

Comparison_Prompt_B_widget = widgets.Textarea(
    value="You are a playful teacher. Use a friendly tone and explain with a daily-life example.",
    description="Prompt B:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="900px", height="95px"),
)

compare_prompts_button = widgets.Button(
    description="Compare responses",
    button_style="primary",
    layout=widgets.Layout(width="180px", height="40px"),
)

compare_status_html = widgets.HTML(
    value="<span style='color:#666;'>Ready.</span>"
)

compare_progress = widgets.IntProgress(
    value=0,
    min=0,
    max=100,
    description="Processing:",
    bar_style="",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px", visibility="hidden"),
)

compare_elapsed_html = widgets.HTML(
    value="<span style='color:#666;'>Elapsed time: 0.0s</span>"
)

comparison_out = widgets.HTML(
    value="""
    <div style="
        width:900px;
        min-height:120px;
        padding:14px 16px;
        border:1px solid #d9d9d9;
        border-radius:10px;
        background:#fafafa;
        color:#444;
        line-height:1.5;
    ">
        Comparison results will appear here.
    </div>
    """
)

def _format_comparison_box(title, text, border_color="#d9d9d9", background="#fafafa"):
    safe_title = html.escape(title)
    safe_text = html.escape(text).replace("\n", "<br>")
    return f"""
    <div style="
        width:100%;
        padding:16px 18px;
        border:1px solid {border_color};
        border-radius:10px;
        background:{background};
        line-height:1.6;
        white-space:normal;
        overflow-wrap:anywhere;
        box-sizing:border-box;
    ">
        <div style="font-weight:700; margin-bottom:10px;">{safe_title}</div>
        <div style="white-space:pre-wrap;">{safe_text}</div>
    </div>
    """

def _format_comparison_layout(response_a, response_b):
    box_a = _format_comparison_box("Response to Prompt A", response_a, "#cfd8ff", "#f7f9ff")
    box_b = _format_comparison_box("Response to Prompt B", response_b, "#cfead6", "#f7fcf8")
    return f"""
    <div style="width:900px; display:grid; grid-template-columns:1fr; gap:14px;">
        {box_a}
        {box_b}
    </div>
    """

def _run_comparison(_=None):
    global Comparison_User_Prompt, Comparison_Prompt_A, Comparison_Prompt_B
    global MAX_NEW_TOKENS, TEMPERATURE, TOP_P, TOP_K, DO_SAMPLE

    Comparison_User_Prompt = Comparison_User_Prompt_widget.value
    Comparison_Prompt_A = Comparison_Prompt_A_widget.value
    Comparison_Prompt_B = Comparison_Prompt_B_widget.value

    # Safe internal defaults
    MAX_NEW_TOKENS = 220
    TEMPERATURE = 0.7
    TOP_P = 0.95
    TOP_K = 50
    DO_SAMPLE = True

    start_time = time.perf_counter()

    compare_prompts_button.disabled = True
    compare_prompts_button.description = "Running..."
    compare_status_html.value = (
        "<span style='color:#1f77b4;'><b>Running...</b> "
        "Please wait while both responses are being generated.</span>"
    )
    compare_progress.layout.visibility = "visible"
    compare_progress.bar_style = "info"
    compare_progress.value = 15
    compare_elapsed_html.value = "<span style='color:#1f77b4;'>Elapsed time: running...</span>"
    comparison_out.value = _format_comparison_box("Comparison", "Generating responses...")

    if model is None or tokenizer is None:
        elapsed = time.perf_counter() - start_time
        compare_status_html.value = (
            "<span style='color:#b22222;'><b>No model is currently loaded.</b> "
            "Please load a model first.</span>"
        )
        compare_progress.bar_style = "danger"
        compare_progress.value = 100
        compare_elapsed_html.value = f"<span style='color:#b22222;'>Elapsed time: {elapsed:.1f}s</span>"
        comparison_out.value = _format_comparison_box(
            "Comparison",
            "No model is currently loaded. Please load a model first.",
            border_color="#f0b4b4",
            background="#fff5f5",
        )
        compare_prompts_button.disabled = False
        compare_prompts_button.description = "Compare responses"
        return

    try:
        compare_progress.value = 35

        response_a = generate_from_chat(
            model=model,
            tokenizer=tokenizer,
            system_instruction=Comparison_Prompt_A,
            user_prompt=Comparison_User_Prompt,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            do_sample=DO_SAMPLE,
        )

        compare_progress.value = 70

        response_b = generate_from_chat(
            model=model,
            tokenizer=tokenizer,
            system_instruction=Comparison_Prompt_B,
            user_prompt=Comparison_User_Prompt,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            do_sample=DO_SAMPLE,
        )

        elapsed = time.perf_counter() - start_time
        compare_progress.value = 100
        compare_progress.bar_style = "success"
        compare_status_html.value = (
            "<span style='color:green;'><b>Done.</b> Both responses were generated successfully.</span>"
        )
        compare_elapsed_html.value = f"<span style='color:green;'>Elapsed time: {elapsed:.1f}s</span>"

        comparison_out.value = _format_comparison_layout(response_a, response_b)

    except Exception as e:
        elapsed = time.perf_counter() - start_time
        compare_progress.value = 100
        compare_progress.bar_style = "danger"
        compare_status_html.value = "<span style='color:#b22222;'><b>Comparison failed.</b></span>"
        compare_elapsed_html.value = f"<span style='color:#b22222;'>Elapsed time: {elapsed:.1f}s</span>"
        comparison_out.value = _format_comparison_box(
            "Error",
            f"Comparison failed.\n\nError type: {type(e).__name__}\n{str(e)}",
            border_color="#f0b4b4",
            background="#fff5f5",
        )

    finally:
        compare_prompts_button.disabled = False
        compare_prompts_button.description = "Compare responses"

compare_prompts_button.on_click(_run_comparison)

display(
    widgets.VBox([
        Comparison_User_Prompt_widget,
        Comparison_Prompt_A_widget,
        Comparison_Prompt_B_widget,
        widgets.HBox([compare_prompts_button]),
        compare_status_html,
        compare_progress,
        compare_elapsed_html,
        comparison_out,
    ])
)

# 2. Prompt Types: Zero-Shot, Few-Shot, and Chain-of-Thought-Style

These are not different models.  
They are different **ways of writing prompts**.

## 2.1 Zero-shot
You ask directly, without examples.

## 2.2 Few-shot
You include a few examples of the desired pattern.

## 2.3 Chain-of-thought-style prompting
You ask the model to work in stages or explain reasoning step by step.

### Practical note
Not every model benefits equally from every prompt type.  
Part of prompt engineering is finding which style works best for the chosen model and task.

In [ ]:
# ==== Explore zero-shot, few-shot, and Prompt C prompting separately ====

import html
import time

Prompt_Task_widget = widgets.Textarea(
    value="Classify the customer review into one of these labels: Positive, Negative, Neutral.\n\nReview: 'The laptop starts quickly and the screen is bright, but the battery life is much shorter than advertised.'",
    description="Task:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="900px", height="130px"),
)

Prompt_A_widget = widgets.Text(
    value="You are a careful assistant who follows instructions exactly and gives concise, accurate answers.",
    description="Prompt A:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="900px"),
)

refresh_prompt_types_button = widgets.Button(
    description="Build prompt types",
    button_style="primary",
    layout=widgets.Layout(width="170px", height="40px"),
)

prompt_types_status_html = widgets.HTML(
    value="<span style='color:#666;'>Ready.</span>"
)

def _format_prompt_box(title, prompt_text, border_color="#d9d9d9", background="#fafafa"):
    safe_title = html.escape(title)
    safe_prompt = html.escape(prompt_text).replace("\n", "<br>")
    return f"""
    <div style="
        width:100%;
        padding:16px 18px;
        border:1px solid {border_color};
        border-radius:10px;
        background:{background};
        line-height:1.6;
        white-space:normal;
        overflow-wrap:anywhere;
        box-sizing:border-box;
    ">
        <div style="font-weight:700; margin-bottom:10px;">{safe_title}</div>
        <div style="white-space:pre-wrap;">{safe_prompt}</div>
    </div>
    """

def _format_response_box(title, text, border_color="#d9d9d9", background="#fafafa"):
    safe_title = html.escape(title)
    safe_text = html.escape(text).replace("\n", "<br>")
    return f"""
    <div style="
        width:100%;
        padding:16px 18px;
        border:1px solid {border_color};
        border-radius:10px;
        background:{background};
        line-height:1.6;
        white-space:normal;
        overflow-wrap:anywhere;
        box-sizing:border-box;
    ">
        <div style="font-weight:700; margin-bottom:10px;">{safe_title}</div>
        <div style="white-space:pre-wrap;">{safe_text}</div>
    </div>
    """

def _running_spinner_html(label_text):
    safe_label = html.escape(label_text)
    return f"""
    <div style="display:flex; align-items:center; gap:10px; color:#1f77b4; margin:6px 0 4px 0;">
        <div style="
            width:18px;
            height:18px;
            border:3px solid #d0d7ff;
            border-top:3px solid #1f77b4;
            border-radius:50%;
            animation: spin 1s linear infinite;
        "></div>
        <div><b>{safe_label}</b></div>
    </div>

    <style>
    @keyframes spin {{
        0% {{ transform: rotate(0deg); }}
        100% {{ transform: rotate(360deg); }}
    }}
    </style>
    """

Zero_Shot_prompt_html = widgets.HTML()
Few_Shot_prompt_html = widgets.HTML()
Prompt_C_prompt_html = widgets.HTML()

zero_shot_run_button = widgets.Button(
    description="Run zero-shot",
    button_style="primary",
    layout=widgets.Layout(width="160px", height="38px"),
)
few_shot_run_button = widgets.Button(
    description="Run few-shot",
    button_style="primary",
    layout=widgets.Layout(width="160px", height="38px"),
)
prompt_c_run_button = widgets.Button(
    description="Run Prompt C",
    button_style="primary",
    layout=widgets.Layout(width="160px", height="38px"),
)

zero_shot_status_html = widgets.HTML(value="<span style='color:#666;'>Ready.</span>")
few_shot_status_html = widgets.HTML(value="<span style='color:#666;'>Ready.</span>")
prompt_c_status_html = widgets.HTML(value="<span style='color:#666;'>Ready.</span>")

zero_shot_progress = widgets.IntProgress(
    value=0, min=0, max=100, description="Processing:",
    bar_style="", style={"description_width": "initial"},
    layout=widgets.Layout(width="500px", visibility="hidden"),
)
few_shot_progress = widgets.IntProgress(
    value=0, min=0, max=100, description="Processing:",
    bar_style="", style={"description_width": "initial"},
    layout=widgets.Layout(width="500px", visibility="hidden"),
)
prompt_c_progress = widgets.IntProgress(
    value=0, min=0, max=100, description="Processing:",
    bar_style="", style={"description_width": "initial"},
    layout=widgets.Layout(width="500px", visibility="hidden"),
)

zero_shot_elapsed_html = widgets.HTML(value="<span style='color:#666;'>Elapsed time: 0.0s</span>")
few_shot_elapsed_html = widgets.HTML(value="<span style='color:#666;'>Elapsed time: 0.0s</span>")
prompt_c_elapsed_html = widgets.HTML(value="<span style='color:#666;'>Elapsed time: 0.0s</span>")

zero_shot_spinner_html = widgets.HTML(value="", layout=widgets.Layout(width="900px"))
few_shot_spinner_html = widgets.HTML(value="", layout=widgets.Layout(width="900px"))
prompt_c_spinner_html = widgets.HTML(value="", layout=widgets.Layout(width="900px"))

zero_shot_response_html = widgets.HTML(
    value=_format_response_box("Zero-shot response", "Response will appear here.", "#cfd8ff", "#f7f9ff")
)
few_shot_response_html = widgets.HTML(
    value=_format_response_box("Few-shot response", "Response will appear here.", "#cfead6", "#f7fcf8")
)
prompt_c_response_html = widgets.HTML(
    value=_format_response_box("Prompt C response", "Response will appear here.", "#f2ddc6", "#fffaf4")
)

ZERO_SHOT_PROMPT = ""
FEW_SHOT_PROMPT = ""
PROMPT_C = ""

def _build_prompt_types(_=None):
    global Prompt_Task, Prompt_A, ZERO_SHOT_PROMPT, FEW_SHOT_PROMPT, PROMPT_C

    Prompt_Task = Prompt_Task_widget.value.strip()
    Prompt_A = Prompt_A_widget.value.strip()

    ZERO_SHOT_PROMPT = Prompt_Task

    FEW_SHOT_PROMPT = (
        "Classify each review into one label: Positive, Negative, or Neutral.\n\n"
        "Review: 'The food arrived hot and tasted amazing.'\n"
        "Label: Positive\n\n"
        "Review: 'The package came late and the item was damaged.'\n"
        "Label: Negative\n\n"
        "Review: 'The hotel room was clean, but nothing stood out as special.'\n"
        "Label: Neutral\n\n"
        f"{Prompt_Task}\n"
        "Label:"
    )

    PROMPT_C = (
        "Classify the review into one label: Positive, Negative, or Neutral.\n"
        "First identify the positive and negative signals in the review.\n"
        "Then decide which overall label fits best.\n"
        "Finally give the answer in this format:\n"
        "Reasoning: <short explanation>\n"
        "Label: <one label>\n\n"
        f"{Prompt_Task}"
    )

    Zero_Shot_prompt_html.value = _format_prompt_box("Zero-shot prompt", ZERO_SHOT_PROMPT, "#cfd8ff", "#f7f9ff")
    Few_Shot_prompt_html.value = _format_prompt_box("Few-shot prompt", FEW_SHOT_PROMPT, "#cfead6", "#f7fcf8")
    Prompt_C_prompt_html.value = _format_prompt_box("Prompt C prompt", PROMPT_C, "#f2ddc6", "#fffaf4")

    prompt_types_status_html.value = (
        "<span style='color:green;'><b>Done.</b> Prompt types are ready. Run any of them separately below.</span>"
    )

def _run_single_prompt(prompt_text, response_widget, status_widget, progress_widget, elapsed_widget, spinner_widget, button_widget, response_title, border_color, background):
    global Prompt_A, MAX_NEW_TOKENS, TEMPERATURE, TOP_P, TOP_K, DO_SAMPLE

    Prompt_A = Prompt_A_widget.value.strip()

    # Safe internal defaults
    MAX_NEW_TOKENS = 180
    TEMPERATURE = 0.4
    TOP_P = 0.95
    TOP_K = 50
    DO_SAMPLE = True

    start_time = time.perf_counter()

    button_widget.disabled = True
    original_description = button_widget.description
    button_widget.description = "Running..."
    status_widget.value = "<span style='color:#1f77b4;'><b>Running...</b> Please wait while the response is being generated.</span>"
    progress_widget.layout.visibility = "visible"
    progress_widget.bar_style = "info"
    progress_widget.value = 35
    elapsed_widget.value = "<span style='color:#1f77b4;'>Elapsed time: running...</span>"
    spinner_widget.value = _running_spinner_html("Generating response...")
    response_widget.value = _format_response_box(response_title, "Generating response...", border_color, background)

    if model is None or tokenizer is None:
        elapsed = time.perf_counter() - start_time
        status_widget.value = "<span style='color:#b22222;'><b>No model is currently loaded.</b> Please load a model first.</span>"
        progress_widget.bar_style = "danger"
        progress_widget.value = 100
        elapsed_widget.value = f"<span style='color:#b22222;'>Elapsed time: {elapsed:.1f}s</span>"
        spinner_widget.value = ""
        response_widget.value = _format_response_box(
            response_title,
            "No model is currently loaded. Please load a model first.",
            "#f0b4b4",
            "#fff5f5",
        )
        button_widget.disabled = False
        button_widget.description = original_description
        return

    try:
        progress_widget.value = 65

        response = generate_from_chat(
            model=model,
            tokenizer=tokenizer,
            system_instruction=Prompt_A,
            user_prompt=prompt_text,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            do_sample=DO_SAMPLE,
        )

        elapsed = time.perf_counter() - start_time
        progress_widget.value = 100
        progress_widget.bar_style = "success"
        status_widget.value = "<span style='color:green;'><b>Done.</b> Response generated successfully.</span>"
        elapsed_widget.value = f"<span style='color:green;'>Elapsed time: {elapsed:.1f}s</span>"
        spinner_widget.value = ""
        response_widget.value = _format_response_box(response_title, response, border_color, background)

    except Exception as e:
        elapsed = time.perf_counter() - start_time
        progress_widget.value = 100
        progress_widget.bar_style = "danger"
        status_widget.value = "<span style='color:#b22222;'><b>Run failed.</b></span>"
        elapsed_widget.value = f"<span style='color:#b22222;'>Elapsed time: {elapsed:.1f}s</span>"
        spinner_widget.value = ""
        response_widget.value = _format_response_box(
            "Error",
            f"Run failed.\n\nError type: {type(e).__name__}\n{str(e)}",
            "#f0b4b4",
            "#fff5f5",
        )

    finally:
        button_widget.disabled = False
        button_widget.description = original_description

def _run_zero_shot(_=None):
    _build_prompt_types()
    _run_single_prompt(
        ZERO_SHOT_PROMPT,
        zero_shot_response_html,
        zero_shot_status_html,
        zero_shot_progress,
        zero_shot_elapsed_html,
        zero_shot_spinner_html,
        zero_shot_run_button,
        "Zero-shot response",
        "#cfd8ff",
        "#f7f9ff",
    )

def _run_few_shot(_=None):
    _build_prompt_types()
    _run_single_prompt(
        FEW_SHOT_PROMPT,
        few_shot_response_html,
        few_shot_status_html,
        few_shot_progress,
        few_shot_elapsed_html,
        few_shot_spinner_html,
        few_shot_run_button,
        "Few-shot response",
        "#cfead6",
        "#f7fcf8",
    )

def _run_prompt_c(_=None):
    _build_prompt_types()
    _run_single_prompt(
        PROMPT_C,
        prompt_c_response_html,
        prompt_c_status_html,
        prompt_c_progress,
        prompt_c_elapsed_html,
        prompt_c_spinner_html,
        prompt_c_run_button,
        "Prompt C response",
        "#f2ddc6",
        "#fffaf4",
    )

refresh_prompt_types_button.on_click(_build_prompt_types)
zero_shot_run_button.on_click(_run_zero_shot)
few_shot_run_button.on_click(_run_few_shot)
prompt_c_run_button.on_click(_run_prompt_c)

_build_prompt_types()

display(
    widgets.VBox([
        Prompt_Task_widget,
        Prompt_A_widget,
        widgets.HBox([refresh_prompt_types_button]),
        prompt_types_status_html,

        Zero_Shot_prompt_html,
        zero_shot_run_button,
        zero_shot_status_html,
        zero_shot_progress,
        zero_shot_elapsed_html,
        zero_shot_spinner_html,
        zero_shot_response_html,

        Few_Shot_prompt_html,
        few_shot_run_button,
        few_shot_status_html,
        few_shot_progress,
        few_shot_elapsed_html,
        few_shot_spinner_html,
        few_shot_response_html,

        Prompt_C_prompt_html,
        prompt_c_run_button,
        prompt_c_status_html,
        prompt_c_progress,
        prompt_c_elapsed_html,
        prompt_c_spinner_html,
        prompt_c_response_html,
    ])
)

## What to observe

As you compare the outputs, ask:

- Which prompt gave the most accurate translation?
- Which gave the cleanest format?
- Did extra examples help?
- Did asking for steps help or did it make the answer unnecessarily long?

This is the start of **iterative prompt refinement**.

# 3. Iterative Refinement

Good prompting is often iterative.

A common workflow is:

1. start with a baseline prompt,
2. inspect the output,
3. identify a weakness,
4. refine one thing,
5. test again,
6. document what improved.

### Common prompt problems
- too vague,
- too long,
- conflicting instructions,
- missing output format,
- unclear audience,
- missing constraints,
- asking for too many things at once.

In [ ]:
# ==== Explore prompt refinement: baseline vs refined versions ====

import html
import time

Prompt_A_widget = widgets.Text(
    value="You are helpful, accurate, and clear.",
    description="Prompt A:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="900px"),
)

Baseline_Prompt_widget = widgets.Textarea(
    value="Summarize the impact of renewable energy.",
    description="Baseline:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="900px", height="80px"),
)

Refined_Prompt_B_widget = widgets.Textarea(
    value="Summarize the impact of renewable energy in 4 bullet points.",
    description="Refined B:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="900px", height="80px"),
)

Refined_Prompt_C_widget = widgets.Textarea(
    value=(
        "You are an energy-policy tutor. Summarize the impact of renewable energy "
        "in exactly 4 bullet points for undergraduate students. "
        "Include one economic effect, one environmental effect, and one limitation."
    ),
    description="Refined C:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="900px", height="110px"),
)

run_baseline_button = widgets.Button(
    description="Run Baseline",
    button_style="primary",
    layout=widgets.Layout(width="150px", height="38px"),
)

run_refined_b_button = widgets.Button(
    description="Run Refined B",
    button_style="primary",
    layout=widgets.Layout(width="160px", height="38px"),
)

run_refined_c_button = widgets.Button(
    description="Run Refined C",
    button_style="primary",
    layout=widgets.Layout(width="160px", height="38px"),
)

run_all_refinement_button = widgets.Button(
    description="Run all",
    button_style="success",
    layout=widgets.Layout(width="120px", height="38px"),
)

refinement_status_html = widgets.HTML(
    value="<span style='color:#666;'>Ready.</span>"
)

def _format_refinement_box(title, text, border_color="#d9d9d9", background="#fafafa"):
    safe_title = html.escape(title)
    safe_text = html.escape(text).replace("\n", "<br>")
    return f"""
    <div style="
        width:100%;
        padding:16px 18px;
        border:1px solid {border_color};
        border-radius:10px;
        background:{background};
        line-height:1.6;
        white-space:normal;
        overflow-wrap:anywhere;
        box-sizing:border-box;
    ">
        <div style="font-weight:700; margin-bottom:10px;">{safe_title}</div>
        <div style="white-space:pre-wrap;">{safe_text}</div>
    </div>
    """

def _running_spinner_html(label_text):
    safe_label = html.escape(label_text)
    return f"""
    <div style="display:flex; align-items:center; gap:10px; color:#1f77b4; margin:6px 0 4px 0;">
        <div style="
            width:18px;
            height:18px;
            border:3px solid #d0d7ff;
            border-top:3px solid #1f77b4;
            border-radius:50%;
            animation: spin 1s linear infinite;
        "></div>
        <div><b>{safe_label}</b></div>
    </div>

    <style>
    @keyframes spin {{
        0% {{ transform: rotate(0deg); }}
        100% {{ transform: rotate(360deg); }}
    }}
    </style>
    """

Baseline_Prompt_html = widgets.HTML()
Refined_Prompt_B_html = widgets.HTML()
Refined_Prompt_C_html = widgets.HTML()

baseline_status_html = widgets.HTML(value="<span style='color:#666;'>Ready.</span>")
refined_b_status_html = widgets.HTML(value="<span style='color:#666;'>Ready.</span>")
refined_c_status_html = widgets.HTML(value="<span style='color:#666;'>Ready.</span>")

baseline_progress = widgets.IntProgress(
    value=0, min=0, max=100, description="Processing:",
    bar_style="", style={"description_width": "initial"},
    layout=widgets.Layout(width="500px", visibility="hidden"),
)
refined_b_progress = widgets.IntProgress(
    value=0, min=0, max=100, description="Processing:",
    bar_style="", style={"description_width": "initial"},
    layout=widgets.Layout(width="500px", visibility="hidden"),
)
refined_c_progress = widgets.IntProgress(
    value=0, min=0, max=100, description="Processing:",
    bar_style="", style={"description_width": "initial"},
    layout=widgets.Layout(width="500px", visibility="hidden"),
)

baseline_elapsed_html = widgets.HTML(value="<span style='color:#666;'>Elapsed time: 0.0s</span>")
refined_b_elapsed_html = widgets.HTML(value="<span style='color:#666;'>Elapsed time: 0.0s</span>")
refined_c_elapsed_html = widgets.HTML(value="<span style='color:#666;'>Elapsed time: 0.0s</span>")

baseline_spinner_html = widgets.HTML(value="", layout=widgets.Layout(width="900px"))
refined_b_spinner_html = widgets.HTML(value="", layout=widgets.Layout(width="900px"))
refined_c_spinner_html = widgets.HTML(value="", layout=widgets.Layout(width="900px"))

baseline_response_html = widgets.HTML(
    value=_format_refinement_box("Baseline response", "Response will appear here.", "#cfd8ff", "#f7f9ff")
)
refined_b_response_html = widgets.HTML(
    value=_format_refinement_box("Refined B response", "Response will appear here.", "#cfead6", "#f7fcf8")
)
refined_c_response_html = widgets.HTML(
    value=_format_refinement_box("Refined C response", "Response will appear here.", "#f2ddc6", "#fffaf4")
)

BASELINE_PROMPT = ""
REFINED_PROMPT_B = ""
REFINED_PROMPT_C = ""

def _refresh_refinement_prompts():
    global Prompt_A, BASELINE_PROMPT, REFINED_PROMPT_B, REFINED_PROMPT_C

    Prompt_A = Prompt_A_widget.value.strip()
    BASELINE_PROMPT = Baseline_Prompt_widget.value.strip()
    REFINED_PROMPT_B = Refined_Prompt_B_widget.value.strip()
    REFINED_PROMPT_C = Refined_Prompt_C_widget.value.strip()

    Baseline_Prompt_html.value = _format_refinement_box("Baseline prompt", BASELINE_PROMPT, "#cfd8ff", "#f7f9ff")
    Refined_Prompt_B_html.value = _format_refinement_box("Refined B prompt", REFINED_PROMPT_B, "#cfead6", "#f7fcf8")
    Refined_Prompt_C_html.value = _format_refinement_box("Refined C prompt", REFINED_PROMPT_C, "#f2ddc6", "#fffaf4")

def _run_single_refinement(prompt_text, response_widget, status_widget, progress_widget, elapsed_widget, spinner_widget, button_widget, response_title, border_color, background):
    global Prompt_A, MAX_NEW_TOKENS, TEMPERATURE, TOP_P, TOP_K, DO_SAMPLE

    Prompt_A = Prompt_A_widget.value.strip()

    # Safe internal defaults
    MAX_NEW_TOKENS = 220
    TEMPERATURE = 0.7
    TOP_P = 0.95
    TOP_K = 50
    DO_SAMPLE = True

    start_time = time.perf_counter()

    button_widget.disabled = True
    original_description = button_widget.description
    button_widget.description = "Running..."
    status_widget.value = "<span style='color:#1f77b4;'><b>Running...</b> Please wait while the response is being generated.</span>"
    progress_widget.layout.visibility = "visible"
    progress_widget.bar_style = "info"
    progress_widget.value = 35
    elapsed_widget.value = "<span style='color:#1f77b4;'>Elapsed time: running...</span>"
    spinner_widget.value = _running_spinner_html("Generating response...")
    response_widget.value = _format_refinement_box(response_title, "Generating response...", border_color, background)

    if model is None or tokenizer is None:
        elapsed = time.perf_counter() - start_time
        status_widget.value = "<span style='color:#b22222;'><b>No model is currently loaded.</b> Please load a model first.</span>"
        progress_widget.bar_style = "danger"
        progress_widget.value = 100
        elapsed_widget.value = f"<span style='color:#b22222;'>Elapsed time: {elapsed:.1f}s</span>"
        spinner_widget.value = ""
        response_widget.value = _format_refinement_box(
            response_title,
            "No model is currently loaded. Please load a model first.",
            "#f0b4b4",
            "#fff5f5",
        )
        button_widget.disabled = False
        button_widget.description = original_description
        return

    try:
        progress_widget.value = 65

        response = generate_from_chat(
            model=model,
            tokenizer=tokenizer,
            system_instruction=Prompt_A,
            user_prompt=prompt_text,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            do_sample=DO_SAMPLE,
        )

        elapsed = time.perf_counter() - start_time
        progress_widget.value = 100
        progress_widget.bar_style = "success"
        status_widget.value = "<span style='color:green;'><b>Done.</b> Response generated successfully.</span>"
        elapsed_widget.value = f"<span style='color:green;'>Elapsed time: {elapsed:.1f}s</span>"
        spinner_widget.value = ""
        response_widget.value = _format_refinement_box(response_title, response, border_color, background)

    except Exception as e:
        elapsed = time.perf_counter() - start_time
        progress_widget.value = 100
        progress_widget.bar_style = "danger"
        status_widget.value = "<span style='color:#b22222;'><b>Run failed.</b></span>"
        elapsed_widget.value = f"<span style='color:#b22222;'>Elapsed time: {elapsed:.1f}s</span>"
        spinner_widget.value = ""
        response_widget.value = _format_refinement_box(
            "Error",
            f"Run failed.\n\nError type: {type(e).__name__}\n{str(e)}",
            "#f0b4b4",
            "#fff5f5",
        )

    finally:
        button_widget.disabled = False
        button_widget.description = original_description

def _run_baseline(_=None):
    _refresh_refinement_prompts()
    refinement_status_html.value = "<span style='color:#1f77b4;'><b>Running baseline prompt...</b></span>"
    _run_single_refinement(
        BASELINE_PROMPT,
        baseline_response_html,
        baseline_status_html,
        baseline_progress,
        baseline_elapsed_html,
        baseline_spinner_html,
        run_baseline_button,
        "Baseline response",
        "#cfd8ff",
        "#f7f9ff",
    )
    refinement_status_html.value = "<span style='color:green;'><b>Done.</b> Baseline prompt finished.</span>"

def _run_refined_b(_=None):
    _refresh_refinement_prompts()
    refinement_status_html.value = "<span style='color:#1f77b4;'><b>Running Refined B prompt...</b></span>"
    _run_single_refinement(
        REFINED_PROMPT_B,
        refined_b_response_html,
        refined_b_status_html,
        refined_b_progress,
        refined_b_elapsed_html,
        refined_b_spinner_html,
        run_refined_b_button,
        "Refined B response",
        "#cfead6",
        "#f7fcf8",
    )
    refinement_status_html.value = "<span style='color:green;'><b>Done.</b> Refined B prompt finished.</span>"

def _run_refined_c(_=None):
    _refresh_refinement_prompts()
    refinement_status_html.value = "<span style='color:#1f77b4;'><b>Running Refined C prompt...</b></span>"
    _run_single_refinement(
        REFINED_PROMPT_C,
        refined_c_response_html,
        refined_c_status_html,
        refined_c_progress,
        refined_c_elapsed_html,
        refined_c_spinner_html,
        run_refined_c_button,
        "Refined C response",
        "#f2ddc6",
        "#fffaf4",
    )
    refinement_status_html.value = "<span style='color:green;'><b>Done.</b> Refined C prompt finished.</span>"

def _run_all_refinements(_=None):
    _refresh_refinement_prompts()
    refinement_status_html.value = "<span style='color:#1f77b4;'><b>Running all refinement stages...</b></span>"
    _run_baseline()
    _run_refined_b()
    _run_refined_c()
    refinement_status_html.value = "<span style='color:green;'><b>Done.</b> All refinement stages completed.</span>"

run_baseline_button.on_click(_run_baseline)
run_refined_b_button.on_click(_run_refined_b)
run_refined_c_button.on_click(_run_refined_c)
run_all_refinement_button.on_click(_run_all_refinements)

_refresh_refinement_prompts()

display(
    widgets.VBox([
        Prompt_A_widget,
        refinement_status_html,

        Baseline_Prompt_widget,
        Baseline_Prompt_html,
        run_baseline_button,
        baseline_status_html,
        baseline_progress,
        baseline_elapsed_html,
        baseline_spinner_html,
        baseline_response_html,

        Refined_Prompt_B_widget,
        Refined_Prompt_B_html,
        run_refined_b_button,
        refined_b_status_html,
        refined_b_progress,
        refined_b_elapsed_html,
        refined_b_spinner_html,
        refined_b_response_html,

        Refined_Prompt_C_widget,
        Refined_Prompt_C_html,
        run_refined_c_button,
        refined_c_status_html,
        refined_c_progress,
        refined_c_elapsed_html,
        refined_c_spinner_html,
        refined_c_response_html,

        widgets.HBox([run_all_refinement_button]),
    ])
)

## Reflection questions for refinement

- Was the baseline answer too broad?
- Did the refined versions improve structure?
- Did adding role + audience + constraints improve quality?
- Was one version more useful?
- Which version would you keep and why?

## 4. Simple local chatbot without widgets

In this section, we build a very simple chatbot using a locally loaded language model.

This version avoids widgets so the logic is easier to understand. The goal is to show the core steps of a chatbot:

1. load a tokenizer and model,
2. define a system instruction,
3. collect user input,
4. generate a response,
5. optionally keep conversation history.

This is useful for learning because it shows the chatbot logic directly in Python.

### What this code demonstrates
- how a chatbot sends a user message to a model,
- how the model uses a system instruction plus conversation history,
- how text generation works at a basic level,
- and how this same logic can later be reused in a web app, mobile app, or server API.


In [ ]:
# ============================================================
# SIMPLE LOCAL CHATBOT (NO WIDGETS, INDEPENDENT VERSION)
# ============================================================
# This example shows the core logic of a chatbot in plain Python.
# It is intentionally simple so the flow is easy to understand.
#
# In this version, the chatbot section is independent from earlier
# notebook sections. That means it loads its own model and tokenizer
# here, instead of depending on previously created variables.
#
# For this section, we use a lightweight model:
#     HuggingFaceTB/SmolLM2-1.7B-Instruct
#
# Why this model?
# - It is compact compared with many other instruction models.
# - It is suitable for lightweight local demos.
# - It is easier to run on limited hardware than larger models.
# ============================================================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ------------------------------------------------------------
# Step 1: Choose a lightweight model for this standalone section
# ------------------------------------------------------------
# This section is independent of earlier notebook code.
# So we define the model ID directly here.
# ------------------------------------------------------------
CHATBOT_MODEL_ID = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

# ------------------------------------------------------------
# Step 2: Detect whether we are using CPU or GPU
# ------------------------------------------------------------
# If CUDA is available, the model can run on GPU.
# Otherwise, it will run on CPU.
# ------------------------------------------------------------
CHATBOT_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Chatbot device:", CHATBOT_DEVICE)

# ------------------------------------------------------------
# Step 3: Load tokenizer
# ------------------------------------------------------------
# The tokenizer converts text into tokens the model can process.
# ------------------------------------------------------------
chatbot_tokenizer = AutoTokenizer.from_pretrained(
    CHATBOT_MODEL_ID,
    trust_remote_code=True,
)

# ------------------------------------------------------------
# Step 4: Make sure a padding token exists
# ------------------------------------------------------------
# Some tokenizer/model combinations do not define a pad token.
# In that case, we use the EOS token as the padding token.
# ------------------------------------------------------------
if chatbot_tokenizer.pad_token is None:
    chatbot_tokenizer.pad_token = chatbot_tokenizer.eos_token

# ------------------------------------------------------------
# Step 5: Load the model
# ------------------------------------------------------------
# We use safe defaults:
# - float16 on CUDA when available
# - float32 on CPU
# ------------------------------------------------------------
chatbot_model_kwargs = {
    "trust_remote_code": True,
}

if CHATBOT_DEVICE == "cuda":
    chatbot_model_kwargs["dtype"] = torch.float16
    chatbot_model_kwargs["device_map"] = "auto"
else:
    chatbot_model_kwargs["dtype"] = torch.float32

chatbot_model = AutoModelForCausalLM.from_pretrained(
    CHATBOT_MODEL_ID,
    **chatbot_model_kwargs,
)

chatbot_model.eval()

print("Lightweight chatbot model loaded successfully.")

# ------------------------------------------------------------
# Step 6: Define the chatbot's system instruction
# ------------------------------------------------------------
# The system instruction gives the chatbot its role and behavior.
# You can think of this as the chatbot's personality and rules.
# ------------------------------------------------------------
CHATBOT_SYSTEM_INSTRUCTION = (
    "You are a helpful university teaching assistant. "
    "Give clear, accurate, structured, beginner-friendly answers. "
    "If something is uncertain, say so honestly."
)

# ------------------------------------------------------------
# Step 7: Store conversation history
# ------------------------------------------------------------
# A chatbot usually needs memory within the current conversation.
# We keep that history in a Python list of messages.
# Each message has:
# - a role ('system', 'user', or 'assistant')
# - content (the text itself)
# ------------------------------------------------------------
chatbot_conversation_history = [
    {"role": "system", "content": CHATBOT_SYSTEM_INSTRUCTION}
]

# ------------------------------------------------------------
# Step 8: Define a function to generate a response
# ------------------------------------------------------------
# This function:
# 1. appends the new user message to the conversation,
# 2. converts the conversation into model input,
# 3. generates a reply,
# 4. decodes the output text,
# 5. stores the assistant reply back into the conversation history.
# ------------------------------------------------------------
def chat_with_local_model(user_message, max_new_tokens=220):
    """
    Generate a chatbot response for a single user message.

    Parameters:
        user_message (str): The user's input message.
        max_new_tokens (int): Maximum number of new tokens to generate.

    Returns:
        str: The assistant's response.
    """

    # Add the user's message to the conversation history.
    chatbot_conversation_history.append({"role": "user", "content": user_message})

    # --------------------------------------------------------
    # Step 8a: Convert the conversation into model-ready input
    # --------------------------------------------------------
    # Many modern chat models support a chat template.
    # That template formats the conversation in the way the model expects.
    #
    # If the tokenizer has a chat template, we use it.
    # Otherwise, we fall back to a simple text format.
    # --------------------------------------------------------
    if hasattr(chatbot_tokenizer, "apply_chat_template") and chatbot_tokenizer.chat_template is not None:
        inputs = chatbot_tokenizer.apply_chat_template(
            chatbot_conversation_history,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        )
    else:
        # Fallback formatting for models without a chat template.
        prompt_text = ""
        for msg in chatbot_conversation_history:
            role = msg["role"].capitalize()
            prompt_text += f"{role}: {msg['content']}\n\n"
        prompt_text += "Assistant:"

        inputs = chatbot_tokenizer(
            prompt_text,
            return_tensors="pt",
            padding=True,
            truncation=True,
        )

    # Move inputs to the same device as the model.
    inputs = {k: v.to(chatbot_model.device) for k, v in inputs.items()}

    # Store the input length so we can later separate old tokens
    # from newly generated tokens.
    input_length = inputs["input_ids"].shape[-1]

    # --------------------------------------------------------
    # Step 8b: Generate the model output
    # --------------------------------------------------------
    # These are safe default generation settings.
    # They keep the output reasonably stable and readable.
    # --------------------------------------------------------
    with torch.no_grad():
        outputs = chatbot_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
            top_k=50,
            repetition_penalty=1.05,
            pad_token_id=(
                chatbot_tokenizer.pad_token_id
                if chatbot_tokenizer.pad_token_id is not None
                else chatbot_tokenizer.eos_token_id
            ),
        )

    # --------------------------------------------------------
    # Step 8c: Decode only the new tokens
    # --------------------------------------------------------
    # The model output contains both the original prompt tokens
    # and the new generated tokens. We only want the new part.
    # --------------------------------------------------------
    new_tokens = outputs[0][input_length:]
    assistant_reply = chatbot_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Save the assistant's reply into the conversation history.
    chatbot_conversation_history.append({"role": "assistant", "content": assistant_reply})

    return assistant_reply

# ------------------------------------------------------------
# Step 9: Example single-turn use
# ------------------------------------------------------------
# This is the simplest possible chatbot interaction.
# ------------------------------------------------------------
example_user_message = "Why does prompt engineering matter? Give three simple reasons."
example_response = chat_with_local_model(example_user_message)

print("USER:")
print(example_user_message)
print("\nASSISTANT:")
print(example_response)

# ------------------------------------------------------------
# Step 10: Optional multi-turn loop
# ------------------------------------------------------------
# Uncomment this block if you want a simple terminal-style chat
# inside the notebook.
#
# Type 'exit' to stop the conversation.
# ------------------------------------------------------------
# while True:
#     user_text = input("You: ").strip()
#     if user_text.lower() in {"exit", "quit", "bye"}:
#         print("Assistant: Goodbye!")
#         break
#
#     reply = chat_with_local_model(user_text)
#     print("Assistant:", reply)

### How this simple chatbot works

This chatbot is built from a few essential parts:

#### 1. Model + tokenizer
- The **tokenizer** converts text into tokens the model can process.
- The **model** predicts the next tokens and generates the reply.

#### 2. System instruction
- The system instruction sets the chatbot's role, tone, and behavior.
- In this example, the chatbot acts like a university teaching assistant.

#### 3. Conversation history
- The conversation is stored as a list of messages.
- This allows the chatbot to remember what was said earlier in the current session.

#### 4. Prompt formatting
- Many chat models need a specific format.
- If the tokenizer supports a chat template, we use it.
- Otherwise, we format the text manually.

#### 5. Text generation
- The model produces new tokens based on the conversation so far.
- These tokens are decoded back into readable text.

#### 6. Multi-turn interaction
- By appending both user and assistant messages into the history list,
  the chatbot can continue the same conversation over multiple turns.


## How a simple chatbot can be programmed on a local computer or server

A simple chatbot on a local machine or server usually has the following architecture:

### Local computer version

This is the easiest place to start.

#### Main components
1. **User interface**
   - command line,
   - notebook,
   - desktop app,
   - or a local web page.

2. **Application logic**
   - receives the user message,
   - keeps conversation history,
   - sends the formatted input to the model,
   - returns the generated reply.

3. **Model runtime**
   - Transformers + PyTorch,
   - or another inference engine.

4. **Model files**
   - stored locally after download,
   - loaded from disk into CPU or GPU memory.

#### Typical flow
- user types a message,
- app adds it to history,
- app formats the prompt,
- model generates text,
- app displays the result.

### Server version

A server version is similar, but the chatbot runs on a machine that other devices connect to.

#### Server architecture
1. **Frontend client**
   - browser page,
   - mobile app,
   - tablet app.

2. **Backend API**
   - usually written in Python with FastAPI or Flask,
   - receives requests like `/chat`,
   - passes the request to the model,
   - returns the response as JSON.

3. **Model service**
   - the backend may load the model directly,
   - or call a separate model-serving process.

4. **Optional database**
   - store users,
   - store chat history,
   - store settings,
   - store logs or analytics.

### Why local vs server matters

**Local** is easier for learning and privacy, but limited by the user’s hardware.

**Server-based** is better when:
- multiple users need access,
- the model is too large for personal devices,
- you want one central deployment.


## How this can later be deployed to tablet, iPad, Android, or iPhone

There are two common deployment paths.

### Option 1: Run the chatbot on a server, access it from mobile devices

This is the most practical path.

#### How it works
- The model runs on a local server or cloud server.
- The tablet or phone only shows the interface.
- The mobile device sends the user message to the server.
- The server generates the answer and sends it back.

#### Advantages
- easier to support larger models,
- easier to update the model,
- same backend can serve browser, tablet, and phone,
- better for classroom or multi-user settings.

#### Typical stack
- **Backend:** Python + FastAPI
- **Frontend:** web app, React app, or mobile app
- **Communication:** HTTP API or WebSocket

### Option 2: Run the model directly on the device

This is possible mainly for very small models.

#### How it works
- The model is converted into a mobile-friendly format.
- The app loads the model directly on the phone or tablet.

#### Challenges
- mobile memory limits,
- slower inference,
- battery consumption,
- more difficult packaging and optimization.

#### This path is usually used for
- tiny local assistants,
- offline demos,
- specialized small models.

### Practical recommendation

For learning and real deployment, the best roadmap is:

1. build the chatbot logic in a notebook,
2. move the logic into a Python script,
3. wrap it in a FastAPI backend,
4. create a simple web interface,
5. open that interface on laptop, tablet, or phone,
6. later package it into a mobile app if needed.


## Simple development roadmap

A good beginner-friendly roadmap is:

### Stage 1: Notebook prototype
- test prompts,
- test model behavior,
- understand the core chatbot loop.

### Stage 2: Python script
- move notebook code into functions,
- keep model loading separate from chat logic,
- make the script reusable.

### Stage 3: Backend API
- create endpoints such as:
  - `/load-model`
  - `/chat`
  - `/reset-history`
- send and receive JSON.

### Stage 4: Frontend
- build a simple chat page,
- show user messages and assistant replies,
- add a typing indicator,
- add conversation reset.

### Stage 5: Deployment
- deploy on a local server, lab machine, or cloud VM,
- access it through browser on iPad/Android/iPhone,
- or later build a dedicated mobile app connected to the same backend.


# 5. Group discussion and reflection tasks

Use the notebook outputs to discuss the following:

1. How much did output quality change when only one line of the prompt changed?
2. Which mattered more in your experiments:
   - context,
   - role,
   - style,
   - examples,
   - or decoding parameters?
3. Which model felt best for:
   - learning,
   - summarization,
   - code help,
   - structured answers?
4. What signs of hallucination or bias did you observe?
5. Which prompts gave the cleanest and most reliable results?

# 6. Next steps and advanced prompt patterns

Once the basics feel natural, the next layer includes:

- retrieval-augmented prompting,
- tool use / function calling,
- structured output prompting,
- prompt templates for production systems,
- fine-tuning and adapter tuning,
- evaluation workflows,
- and human feedback loops.

### Important idea
Prompt engineering is usually strongest when combined with:
- good model selection,
- good context retrieval,
- output validation,
- and task-specific evaluation.

# 7. Ethical usage and warnings

## Responsible use principles
- verify important outputs,
- do not treat generated text as automatically true,
- add disclaimers when appropriate,
- check for bias in high-stakes domains,
- avoid using model outputs as sole evidence for medical, legal, or safety-critical decisions.

## Transparency
If you use AI-generated text in coursework, reports, or professional settings:
- disclose the tool appropriately if required,
- verify claims,
- and document how the output was checked.

## Realistic expectation
A good prompt improves model behavior.  
It does **not** make the model infallible.

# 8. Suggested exercises

1. Run the same prompt on **three different models** from the catalog and compare:
   - relevance,
   - clarity,
   - speed,
   - and style.

2. Write one prompt in:
   - zero-shot form,
   - few-shot form,
   - and chain-of-thought-style form.

3. Create a role-based prompt for three personas:
   - teacher,
   - journalist,
   - software engineer.

4. Test how the answer changes when you modify only:
   - the system instruction,
   - the temperature,
   - or the requested output format.

5. Try a code task with `Qwen2.5-Coder-7B-Instruct` and compare it with a general-purpose model.

6. Design a safer prompt for a topic where hallucinations are likely.

# 9. External resources for further study

## Official documentation
- Hugging Face Transformers documentation
- Hugging Face chat templating guide
- Hugging Face pipelines documentation

## Model cards
Review the model card for whichever model you selected:
- Qwen
- Mistral
- Llama
- Phi
- Gemma
- SmolLM

## Recommended concepts to explore next
- chat templates,
- structured output prompting,
- tool calling,
- RAG,
- evaluation rubrics,
- safety prompting,
- fine-tuning versus prompting.